# Thử sinh mô tả chi tiết bằng Qwen2.5-VL-7B (20 ảnh) — GPU L4

Mục đích: trước khi chạy cả 5000 ảnh (~1–2 giờ), kiểm tra xem model có **thấy và ghi ra** các chi tiết mà đề thi hỏi
không: khăn cam trên vòi nước, người áo trắng đội mũ ngồi sofa, thớt đỏ, áo thể thao xanh, đeo kính…

20 ảnh trong `anh_thu_20.zip`: 4 ảnh đáp án đề 23/11, 6 ảnh đội đã nộp sai, 8 ảnh liền kề đáp án (dễ nhầm), 2 ảnh ngẫu nhiên.

**Cách chạy:** Runtime → Change runtime type → **L4 GPU**. Chạy lần lượt từng ô. Nếu ô cài đặt báo cần *Restart session*,
bấm restart rồi chạy tiếp từ ô 3. Cuối cùng tải file `mo_ta_thu_20.jsonl` về và gửi lại để đọc kết quả.

In [ ]:
# 1. Kiểm tra GPU (cần thấy NVIDIA L4, ~23 GB)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Cài vLLM (chạy model theo lô, nhanh hơn ~10 lần so với từng ảnh). Mất ~3–5 phút.
# nvidia-cutlass-dsl: bản vLLM mới lúc khởi động nạp module cần gói "cutlass" nhưng không tự cài
# (lỗi "ModuleNotFoundError: No module named 'cutlass'"). Cài xong nếu Colab báo Restart session -> restart, chạy tiếp từ ô 3.
!pip install -q "vllm>=0.8.5" qwen-vl-utils nvidia-cutlass-dsl

In [ ]:
# 3. Tải lên anh_thu_20.zip (trong thư mục colab/ của repo) rồi giải nén
import os, zipfile, csv
from google.colab import files
if not os.path.exists("anh_thu_20.zip"):
    files.upload()
zipfile.ZipFile("anh_thu_20.zip").extractall("anh_thu")
loai = {r["file"]: r["loai"] for r in csv.DictReader(open("anh_thu/danh_sach.csv", encoding="utf-8"))}
anh = sorted(loai)
print(len(anh), "ảnh:", anh)

In [ ]:
# 4. Nạp Qwen2.5-VL-7B (~16 GB, tải lần đầu ~3–5 phút). Giữ ảnh gần nguyên 1280x720 để thấy vật nhỏ.
#    Thử vLLM trước (nhanh). Nếu vLLM lỗi khi khởi động -> tự chuyển sang transformers (chậm hơn ~5–10 lần,
#    20 ảnh vẫn chỉ vài phút; cùng model nên mô tả gần như giống hệt).
#    Nếu ô này báo hết bộ nhớ GPU (CUDA out of memory) sau khi vLLM lỗi: Runtime -> Restart session, đặt
#    DUNG_VLLM = False rồi chạy lại từ ô 3.
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
DUNG_VLLM = True
llm = None
if DUNG_VLLM:
    try:
        from vllm import LLM, SamplingParams
        llm = LLM(model=MODEL, dtype="bfloat16", max_model_len=8192, gpu_memory_utilization=0.90,
                  limit_mm_per_prompt={"image": 1},
                  mm_processor_kwargs={"min_pixels": 256 * 28 * 28, "max_pixels": 1280 * 720})
        print("Đang dùng vLLM")
    except Exception as e:
        print("vLLM lỗi, chuyển sang transformers:", repr(e)[:300])
        llm = None
if llm is None:
    import gc, torch
    gc.collect(); torch.cuda.empty_cache()
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    hf_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL, torch_dtype=torch.bfloat16, device_map="cuda", attn_implementation="sdpa").eval()
    hf_proc = AutoProcessor.from_pretrained(MODEL, min_pixels=256 * 28 * 28, max_pixels=1280 * 720)
    hf_proc.tokenizer.padding_side = "left"      # sinh chữ theo lô cần đệm bên trái
    print("Đang dùng transformers")

In [ ]:
# 5. Câu lệnh: bắt model tả TỪNG NGƯỜI và TỪNG KHU VỰC (đúng kiểu chi tiết thầy hay hỏi), trả JSON.
PROMPT = """This is a frame from a fixed overhead fisheye camera in a home kitchen. In the back-left there is a living
room with a sofa covered by a red plaid blanket. In the middle there is a kitchen island with a black countertop and a
sink with a faucet. On the right there is a black stove. Describe ONLY what is actually visible; do not guess.

Return valid JSON with exactly these keys:
{
  "people_count": <integer, count everyone including small or partly hidden people in the living room or behind the stove>,
  "people": [
    {
      "where": "<one of: sink on the island, kitchen island, behind the island, stove, dining table, living room sofa, left counter, near the camera, fridge, walkway>",
      "frame_position": "<e.g. bottom-left, center, top-right>",
      "posture": "<standing | sitting | bending | walking>",
      "person": "<man or woman, hair (e.g. long black hair, curly black hair, bald)>",
      "top": "<colour + garment, e.g. blue long-sleeve shirt, white t-shirt, cream sweater, blue sports jersey, sweater with horizontal stripes>",
      "bottom": "<colour + garment, e.g. black skirt, jeans>",
      "accessories": "<e.g. black baseball cap, head-mounted camera on forehead, glasses, apron; or none>",
      "holding": "<objects in hands with colour, e.g. red cutting board, phone; or nothing>",
      "action": "<what they are doing, e.g. washing hands at the sink, looking at a phone>",
      "facing": "<toward the camera | back to the camera | sideways>"
    }
  ],
  "objects": {
    "island_and_sink": "<items with colours, e.g. orange towel hanging on the faucet, pineapple, red cutting board>",
    "stove": "<pots, pans, or empty>",
    "living_room": "<people and objects on or near the sofa>",
    "floor_and_other": "<e.g. two black trash bins at the end of the island>"
  },
  "caption": "<2-4 sentences that mention every person (clothes, action, where) and small coloured objects>"
}"""

In [ ]:
# 6. Chạy theo lô cả 20 ảnh. temperature=0: luôn chọn từ chắc nhất -> không bịa, chạy lại ra cùng kết quả.
import base64, json, re, time
def tin_nhan(path):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    return [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        {"type": "text", "text": PROMPT}]}]
def chay_transformers(ds, lo=4):
    from PIL import Image
    import torch
    kq = []
    for i in range(0, len(ds), lo):
        batch = ds[i:i + lo]
        msg = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
        texts = [hf_proc.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)] * len(batch)
        imgs = [Image.open(f"anh_thu/{f}").convert("RGB") for f in batch]
        inp = hf_proc(text=texts, images=imgs, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            g = hf_model.generate(**inp, max_new_tokens=1200, do_sample=False)
        kq += hf_proc.batch_decode(g[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"  xong {min(i + lo, len(ds))}/{len(ds)}")
    return kq

t = time.time()
if llm is not None:
    sp = SamplingParams(temperature=0, max_tokens=1200)
    van_ban = [o.outputs[0].text for o in llm.chat([tin_nhan(f"anh_thu/{f}") for f in anh], sp)]
else:
    van_ban = chay_transformers(anh)
giay = time.time() - t
print(f"{len(anh)} ảnh trong {giay:.0f} giây = {giay / len(anh):.1f} giây/ảnh ({'vLLM' if llm is not None else 'transformers'})")
print(f"Ước lượng 5000 ảnh với cách này: ~{giay / len(anh) * 5000 / 3600:.1f} giờ")

ket_qua = []
for f, txt in zip(anh, van_ban):
    m = re.search(r"\{.*\}", txt, re.S)
    try:
        js = json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        js = None
    ket_qua.append({"file": f, "loai": loai[f], "json_ok": js is not None, "data": js, "raw": txt})
with open("mo_ta_thu_20.jsonl", "w", encoding="utf-8") as fh:
    for r in ket_qua:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print("JSON hợp lệ:", sum(r["json_ok"] for r in ket_qua), "/", len(ket_qua))

In [ ]:
# 7. Tự kiểm tra 4 ảnh đáp án: mô tả có nhắc các chi tiết quyết định của đề không? (✓ / ✗)
CAN_THAY = {
    "12_320.jpg": {"so_nguoi": 3, "chi_tiet": {
        "rửa tay": r"wash\w* (his |their )?hands", "khăn cam": r"orange (towel|cloth|rag)", "vòi nước": r"faucet|tap",
        "sofa": r"sofa|couch", "áo trắng": r"white (t-)?shirt|white top", "mũ lưỡi trai": r"\bcap\b",
        "camera trên trán": r"head[- ]?(mounted )?camera|camera on (his )?forehead|headlamp", "tóc xoăn": r"curly"}},
    "16_2624.jpg": {"so_nguoi": 5, "chi_tiet": {
        "áo thể thao xanh": r"blue (sports )?jersey|blue sports", "thớt": r"cutting board|chopping board",
        "thớt đỏ": r"(red|orange) (cutting|chopping) board", "áo sọc ngang": r"stripe", "quay lưng": r"back to the camera"}},
    "18_939.jpg": {"so_nguoi": 1, "chi_tiet": {
        "điện thoại": r"phone", "áo len kem": r"cream|beige|off-white", "váy đen": r"black (skirt|dress)",
        "tóc dài": r"long hair", "thùng rác": r"trash|bin"}},
    "15_2827.jpg": {"so_nguoi": 1, "chi_tiet": {
        "điện thoại": r"phone", "đeo kính": r"glasses", "áo len xanh đậm": r"(navy|dark blue) (sweater|jumper)",
        "góc dưới trái": r"bottom[- ]left|lower[- ]left"}},
}
for r in ket_qua:
    if r["file"] not in CAN_THAY:
        continue
    want = CAN_THAY[r["file"]]
    text = json.dumps(r["data"], ensure_ascii=False).lower() if r["data"] else r["raw"].lower()
    n = r["data"].get("people_count") if r["data"] else None
    print(f"\n=== {r['file']} ({r['loai']})  số người model đếm: {n}  (thật: {want['so_nguoi']})")
    for ten, pat in want["chi_tiet"].items():
        print(f"   {'✓' if re.search(pat, text) else '✗'} {ten}")
    if r["data"]:
        print("   caption:", r["data"].get("caption"))

In [ ]:
# 8. Xem toàn bộ mô tả: ảnh đáp án có được tả khác rõ so với ảnh liền kề / ảnh đã nộp sai không?
from IPython.display import Image as IPImage, display
for r in ket_qua:
    print(f"\n#### {r['file']}  [{r['loai']}]")
    display(IPImage(f"anh_thu/{r['file']}", width=480))
    print(json.dumps(r["data"], ensure_ascii=False, indent=1) if r["data"] else r["raw"])

In [ ]:
# 9. Tải kết quả về máy, rồi gửi lại file này
files.download("mo_ta_thu_20.jsonl")